# Rodic_01_CICD_Pipeline
Creates and pushes a GitHub Actions CI/CD pipeline for `Rodic-infra-Colab`, checks it locally first, then reads the run status back into Colab.

**Pipeline design (starter, tighten later)**
- **CI** on every push/PR to `main`: install, lint (syntax and undefined names only, so it starts green), secret-pattern check, pytest.
- **CD**: Streamlit Cloud already redeploys `v2/app_v2.py` automatically when `main` changes, so no deploy step is needed there. Pushing a `v*` tag builds a zip and creates a GitHub Release.
- Not included yet: model-accuracy gates and edge-device (Jetson) tests, since real checkpoints and hardware are not wired in.

Token needs Contents and Workflows read/write plus Actions read. Reuse the `GITHUB_TOKEN` secret from notebook 00.

### Config and authentication helpers
Same helper block as notebook 00 so this notebook runs on its own.

In [ ]:
import os, re, json, base64, shutil, subprocess, pathlib, datetime
from google.colab import userdata

# ---------- CONFIG: edit only this block ----------
REPO         = "Subhash-Pal/Rodic-infra-Colab"
BRANCH       = "main"
VERSION_DIR  = "v4"                 # folder inside the repo that receives your work (Vx/ rule)
WORK_DIR     = "/content/work"      # folder in THIS Colab runtime whose files get pushed
NOTEBOOK_PATH = ""                  # optional: Drive path of the Colab you are editing, e.g.
                                    # "/content/drive/MyDrive/Colab Notebooks/Rodic_Edge_CV.ipynb"
RUN_TESTS    = True                 # run pytest before pushing, if tests exist
DRY_RUN      = False                # True = stage + show diff, no commit/push
MAX_FILE_MB  = 50
# --------------------------------------------------

REPO_DIR = "/content/_repo"
TOKEN = userdata.get("GITHUB_TOKEN")
_B64 = base64.b64encode(f"x-access-token:{TOKEN}".encode()).decode()
AUTH = ["-c", f"http.extraheader=AUTHORIZATION: basic {_B64}"]

def _mask(t): return t.replace(TOKEN, "***").replace(_B64, "***")

def git(*args, cwd=REPO_DIR, auth=False, check=True):
    cmd = ["git"] + (AUTH if auth else []) + list(args)
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    out = _mask(r.stdout + r.stderr).strip()
    if check and r.returncode != 0:
        raise RuntimeError(f"git {args[0]} failed:\n{out}")
    return out

### Clone or update the repo

In [ ]:
if not os.path.isdir(f"{REPO_DIR}/.git"):
    git("clone", "--branch", BRANCH, f"https://github.com/{REPO}.git", REPO_DIR, cwd="/content", auth=True)
else:
    git("fetch", "origin", BRANCH, auth=True)
    git("checkout", BRANCH)
    git("pull", "--rebase", "origin", BRANCH, auth=True)
git("config", "user.name", "Subhash Pal (Colab)")
git("config", "user.email", "colab-sync@users.noreply.github.com")
print(git("log", "--oneline", "-5"))

### Write CI and release workflow files
Creates `.github/workflows/ci.yml` (lint, secret check, pytest) and `release.yml` (zip + GitHub Release on `v*` tags).

In [ ]:
wf = pathlib.Path(REPO_DIR) / ".github" / "workflows"
wf.mkdir(parents=True, exist_ok=True)

(wf / "ci.yml").write_text(r"""name: CI
on:
  push:
    branches: [main]
  pull_request:
jobs:
  test:
    runs-on: ubuntu-latest
    strategy:
      matrix:
        python-version: ["3.11"]
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: ${{ matrix.python-version }}
          cache: pip
      - name: Install
        run: |
          pip install -r v2/requirements.txt pytest pyyaml scikit-image scipy ruff
      - name: Lint (syntax and undefined names only)
        run: ruff check --select E9,F63,F7,F82 .
      - name: Secret pattern check
        run: |
          if grep -RIEn --exclude-dir=.git 'AIza[0-9A-Za-z_-]{35}|ghp_[A-Za-z0-9]{36}|sk-ant-[A-Za-z0-9_-]{20,}' . ; then
            echo "Possible secret committed"; exit 1; fi
      - name: Tests
        run: pytest -q
""")
(wf / "release.yml").write_text(r"""name: Release
on:
  push:
    tags: ["v*"]
permissions:
  contents: write
jobs:
  release:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Package
        run: |
          zip -r rodic-${{ github.ref_name }}.zip v2 v3 README.md -x "*/__pycache__/*"
      - name: Create GitHub Release
        uses: softprops/action-gh-release@v2
        with:
          files: rodic-${{ github.ref_name }}.zip
          generate_release_notes: true
""")
print("Wrote:", *sorted(p.name for p in wf.iterdir()))

### Local dry-run of CI
Runs lint and pytest here with the same commands the workflow uses, so the first GitHub run does not start red.

In [ ]:
# Run the same checks the CI will run, before pushing.
subprocess.run("pip -q install ruff pytest pyyaml scikit-image scipy", shell=True)
if os.path.exists(f"{REPO_DIR}/v2/requirements.txt"):
    subprocess.run(f"pip -q install -r {REPO_DIR}/v2/requirements.txt", shell=True)
lint = subprocess.run("ruff check --select E9,F63,F7,F82 .", shell=True, cwd=REPO_DIR, capture_output=True, text=True)
print("LINT:", "OK" if lint.returncode == 0 else "FAIL\n" + lint.stdout[-1500:])
t = subprocess.run("python -m pytest -q", shell=True, cwd=REPO_DIR, capture_output=True, text=True)
print("TESTS:", "OK" if t.returncode == 0 else "FAIL")
print(t.stdout[-1500:])
LOCAL_OK = (lint.returncode == 0 and t.returncode == 0)

### FINAL CELL: commit and push the workflows
Pushes only `.github/workflows/`. Blocked if the local dry-run failed.

In [ ]:
if not LOCAL_OK:
    print("BLOCKED: local CI dry-run failed. Fix first, otherwise the pipeline would start red.")
else:
    git("add", ".github/workflows")
    if not git("status", "--short"):
        print("Workflows already up to date.")
    else:
        git("commit", "-m", "ci: add CI (lint, secret check, pytest) and tag-based release workflows")
        git("pull", "--rebase", "origin", BRANCH, auth=True)
        git("push", "origin", BRANCH, auth=True)
        print("Pushed workflows. If this failed with a 'workflow scope' error, add Workflows read/write to your token.")

### Check GitHub Actions run status
Polls the Actions API and prints the latest runs with links.

In [ ]:
import time, urllib.request
def latest_runs(n=3):
    req = urllib.request.Request(f"https://api.github.com/repos/{REPO}/actions/runs?per_page={n}",
        headers={"Authorization": f"Bearer {TOKEN}", "Accept": "application/vnd.github+json"})
    return json.load(urllib.request.urlopen(req))["workflow_runs"]

for _ in range(12):                      # poll up to ~2 minutes
    runs = latest_runs()
    if runs and runs[0]["status"] == "completed": break
    time.sleep(10)
for r in runs:
    print(f'{r["name"]:<10} {r["status"]:<12} {r["conclusion"]}  {r["head_sha"][:7]}  {r["html_url"]}')

### Optional: tag a release
Set `TAG` (e.g. `v2.1.0`) to trigger the release workflow.

In [ ]:
# Optional: cut a release. Set TAG and run. This triggers release.yml (zip + GitHub Release).
TAG = ""        # e.g. "v2.1.0"
if TAG:
    git("tag", TAG)
    git("push", "origin", TAG, auth=True)
    print("Tag pushed:", TAG)
else:
    print("TAG empty - skipped.")